---
### Follow the Workers: v2 analysis (diagnostics, v2 variants, forward test)
---

---
##### Imports
Project modules; the frozen study is only ever imported read-only inside data.py.

In [1]:
import hashlib
import json
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import params as P
import data as D
import stats as S
import plots as G

# numpy 2.0 with Apple Accelerate raises spurious "encountered in matmul" warnings; Gate 1 below
# shows results match the frozen study to ~1e-16, so they are silenced (only that message).
warnings.filterwarnings('ignore', message='.*encountered in matmul', category=RuntimeWarning)
pd.set_option('display.width', 160)
for folder in [P.OUT, P.TABLES, P.FIGS, P.TEX, P.FORWARD]:
    folder.mkdir(parents=True, exist_ok=True)


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()


def stop(reason):
    """A failed gate: record it and halt the notebook (CLAUDE.md rule 6)."""
    (P.OUT / 'STOP.md').write_text(f'# STOP\n\n{utc_now()}\n\n{reason}\n')
    raise RuntimeError(reason)


def frozen_fingerprint():
    files = sorted(p for p in P.FROZEN_REPO.rglob('*') if p.is_file() and '.git' not in p.relative_to(P.FROZEN_REPO).parts)
    h = hashlib.sha256()
    for p in files:
        h.update(str(p.relative_to(P.FROZEN_REPO)).encode())
        h.update(hashlib.sha256(p.read_bytes()).digest())
    return h.hexdigest(), len(files)

---
##### Data
Public French returns, current FRED labor data with fixed publication lags, and the frozen study's saved outputs.

In [2]:
R = D.load_returns()                 # 13 group returns, excess, relative, factors
F, T = D.signals()                   # standardised F1..F5, W and tightness T, by earning month
F['A0'] = (F['F1'] + F['F2'] - F['F4'] + F['F5']) / 4      # preregistered fixed rule
F['IndMom'] = D.industry_momentum(R['total'])             # industry momentum 12-1 (months m-12..m-2)
FP, FD = D.frozen()                  # frozen params.py and data.py (read-only)
print('returns', R['total'].index.min(), '->', R['total'].index.max(),
      '| signals', F['W'].dropna(how='all').index.min(), '->', F['W'].dropna(how='all').index.max())

returns 1963-07 -> 2026-08 | signals 2004-04 -> 2026-11


In [3]:
# Which data vintage is used (ITERATION_PROMPT Part 1 step 2), with the evidence for the choice.
panels_in_repo = sorted(str(p.relative_to(P.FROZEN_REPO)) for p in P.FROZEN_REPO.rglob('*.parquet'))
env_key = (P.ROOT / '.env').exists() and 'FRED_API_KEY' in (P.ROOT / '.env').read_text()
vintage_check = {}
for variant in ['FIXEDLAG', 'ASOF', 'REVISED', 'FIRST']:
    saved = D.saved_csv(f'{variant}_A0_scores.csv')
    ours = F['A0'].reindex(saved.index)
    vintage_check[variant] = {
        'pooled_corr': float(np.corrcoef(ours.to_numpy().ravel(), saved.to_numpy().ravel())[0, 1]),
        'mean_monthly_rank_corr': float(S.rank_ic(ours, saved).mean()),
        'same_top3_share': float(np.mean([set(ours.loc[m].nlargest(3).index) == set(saved.loc[m].nlargest(3).index) for m in saved.index]))}
vintage = pd.DataFrame(vintage_check).T
vintage.index.name = 'saved A0 variant'
res = json.loads((P.SEALED / 'results.json').read_text())
ic_a3 = {v: res['variants'][v]['A3']['IC']['1']['mean'] for v in ['ASOF', 'FIXEDLAG']}
choice = 'a' if panels_in_repo else 'b' if env_key else 'c'
text = f"""# Data vintage used in this iteration

Written by analysis/run.ipynb on {utc_now()}.

**Choice: option ({choice})** of ITERATION_PROMPT Part 1 step 2.

- (a) As-known (ASOF) panels saved in the frozen repo: **not available**. The repo contains {len(panels_in_repo)} parquet files; its package_manifest.json lists the panels as omitted.
- (b) ALFRED vintages with `analysis/.env` FRED_API_KEY: **not available** (no key file).
- (c) Revised FRED data downloaded 3 Oct 2026 from fredgraph.csv, with fixed publication lags (JOLTS month d-2, CES month d-1, tightness month d-2): **used** for every historical result in this iteration.

The forward-test book (Part 3) does not use (c). It uses ALFRED values *as published on {P.VINTAGE_DATE}*, downloaded without a key from alfredgraph.csv, so it only contains data known before the 30 Sep 2026 decision.

## How close is (c) to the frozen study's own signals?

Our rebuilt A0 composite (same transforms) against the frozen study's saved sealed A0 scores, Nov 2014 to Aug 2026:

{G.markdown(vintage.round(3))}

Our data is the FIXEDLAG construction (revised values, fixed lags), and it reproduces the frozen FIXEDLAG scores almost exactly. As-known (ASOF) data picks the same top three industries in only {vintage_check['ASOF']['same_top3_share']:.0%} of months, so revisions do change portfolio membership. The frozen study's primary A3 had the same test IC under ASOF ({ic_a3['ASOF']:+.4f}) and FIXEDLAG ({ic_a3['FIXEDLAG']:+.4f}) (results.json), so the conclusions are not sensitive to this choice, but monthly holdings are. Every historical number in this iteration is therefore labelled as computed on revised data.
"""
(P.OUT / 'data_vintage.md').write_text(text)
vintage.round(4)

,pooled_corr,mean_monthly_rank_corr,same_top3_share
saved A0 variant,,,
FIXEDLAG,0.9983,0.9922,0.9085
ASOF,0.9181,0.8635,0.4155
REVISED,0.9912,0.9793,0.8592
FIRST,0.8506,0.7796,0.2394


---
##### Gate 1
Reproduce the sealed A0 net returns from its saved scores with the frozen backtest, and the 13 group returns against every saved ledger.

In [4]:
checks = []

# 1. The frozen repo is unchanged since the baseline fingerprint.
fingerprint, n_files = frozen_fingerprint()
checks.append(('Frozen repo unchanged (SHA-256 over all files)', f'{n_files} files, {fingerprint[:12]}...',
               f'{P.FROZEN_FINGERPRINT[1]} files, {P.FROZEN_FINGERPRINT[0][:12]}...', (fingerprint, n_files) == P.FROZEN_FINGERPRINT))

# 2. Portfolio parameters equal the frozen study's.
same = all(FP.PORTFOLIO[k] == v for k, v in P.PORTFOLIO.items())
checks.append(('Portfolio parameters equal frozen params.PORTFOLIO', str(P.PORTFOLIO), 'identical', same))

# 3. Sealed A0 net monthly returns from the saved A0 scores, through the frozen backtest.
saved_ledger = D.saved_csv('ASOF_A0_ledger.csv')
ledger, _ = D.backtest(D.saved_csv('ASOF_A0_scores.csv'), R)
for col in ['net', 'gross', 'cost', 'turnover']:
    diff = float((ledger[col] - saved_ledger[col]).abs().max())
    checks.append((f'Sealed A0 {col}: max |ours - saved| over {len(ledger)} months', f'{diff:.2e}', '< 1e-6', diff < 1e-6))

# 4. Group returns: saved weights x our group excess returns + hedge = saved gross, every saved ledger.
worst, count = 0.0, 0
for study in [P.SONNET, P.OPUS]:
    sealed = sorted((study / 'outputs/sealed/initial').glob('*_weights.csv'))
    research = sorted((study / 'outputs').glob('research_A*_weights.csv'))
    for path in sealed + research:
        folder = 'outputs/sealed/initial' if path in sealed else 'outputs'
        w = D.saved_csv(path.name, study, folder)
        g = D.saved_csv(path.name.replace('_weights', '_ledger'), study, folder)
        groups = [c for c in w.columns if c != 'market']
        recon = (w[groups] * R['excess'].loc[w.index, groups]).sum(axis=1) + w['market'] * R['factors'].loc[w.index, 'Mkt-RF']
        worst = max(worst, float((recon - g.gross).abs().max()))
        count += 1
checks.append((f'13 group returns vs {count} saved ledgers (both studies, sealed + research)', f'{worst:.2e}', '< 1e-6', worst < 1e-6))

# 5. Our estimators reproduce the frozen results.json (IC, HAC t, Sharpe, factor regression).
res = json.loads((P.SEALED / 'results.json').read_text())
worst_est = 0.0
for s in ['A0', 'A1', 'A1-T', 'A3', 'A4']:
    l, sc = D.saved_csv(f'ASOF_{s}_ledger.csv'), D.saved_csv(f'ASOF_{s}_scores.csv')
    ic = S.rank_ic(sc, R['relative'])
    fa = S.factor_attribution(l.net, R)
    ref, ref_a = res['variants']['ASOF'][s], res['attribution'][s]
    diffs = [float((ic - D.saved_csv(f'ASOF_{s}_ic.csv')['ic']).abs().max()),
             abs(S.hac(ic, lags=6)['t'] - ref['IC']['1']['t']), abs(S.perf(l)['sharpe'] - ref['sharpe']),
             max(abs(fa['coef'][k] - ref_a['coefficients'][k]) for k in ref_a['coefficients']),
             max(abs(fa['tstat'][k] - ref_a['t_statistics'][k]) for k in ref_a['t_statistics'])]
    worst_est = max(worst_est, max(diffs))
checks.append(('Estimators vs results.json (IC, IC t, Sharpe, factor coefs and t; A0-A4)', f'{worst_est:.2e}', '< 1e-8', worst_est < 1e-8))

gate1 = pd.DataFrame(checks, columns=['check', 'value', 'threshold', 'pass'])
gate1

,check,value,threshold,pass
0,Frozen repo unchanged (SHA-256 over all files),"3067 files, eb109be96917...","3067 files, eb109be96917...",True
1,Portfolio parameters equal frozen params.PORTF...,"{'longs': 3, 'shorts': 3, 'holding': 3, 'risk_...",identical,True
2,Sealed A0 net: max |ours - saved| over 142 months,9.97e-17,< 1e-6,True
3,Sealed A0 gross: max |ours - saved| over 142 m...,9.71e-17,< 1e-6,True
4,Sealed A0 cost: max |ours - saved| over 142 mo...,9.94e-17,< 1e-6,True
5,Sealed A0 turnover: max |ours - saved| over 14...,2.22e-16,< 1e-6,True
6,13 group returns vs 50 saved ledgers (both stu...,1.51e-16,< 1e-6,True
7,"Estimators vs results.json (IC, IC t, Sharpe, ...",3.55e-15,< 1e-8,True


In [5]:
passed = bool(gate1['pass'].all())
lines = [f'# Gate 1 validation\n\nRun {utc_now()} by analysis/run.ipynb. **Result: {"PASS" if passed else "FAIL"}**\n',
         'Every check uses the frozen study\'s saved files in `230ga-follow-the-workers/` (read-only) and public Ken French data '
         '(CRSP 202608 build) in `analysis/data/french/`. The backtest is the frozen `data.backtest`, imported read-only.\n',
         G.markdown(gate1, index=False), '',
         '## Informational: rebuilt labor signals vs the frozen study\'s saved A0 scores', '',
         'Not part of the gate (our labor data is revised FRED, not the frozen as-known panels). See `data_vintage.md`.', '',
         G.markdown(vintage.round(4)), '']
(P.OUT / 'validation.md').write_text('\n'.join(lines))
if not passed:
    stop('Gate 1 failed. See analysis/output/validation.md:\n\n' + G.markdown(gate1, index=False))
print('Gate 1:', 'PASS' if passed else 'FAIL')

Gate 1: PASS


---
##### Portfolio equation
The frozen score-to-weight mapping, written as LaTeX for the report.

In [6]:
equation = r"""% Score-to-weight mapping of the frozen study's backtest (230ga-follow-the-workers/outputs/follow_the_workers/data.py::backtest).
% Written by analysis/run.ipynb. Parameters: analysis/params.py PORTFOLIO (checked equal to the frozen params in Gate 1).
\begin{align}
\tau_{m,g} &= \begin{cases} +\tfrac13 & g \in \text{top 3 of } s_{m,\cdot}\\ -\tfrac13 & g \in \text{bottom 3 of } s_{m,\cdot}\\ 0 & \text{otherwise}\end{cases}
  \qquad b_{m} = \frac{1}{H}\sum_{j=0}^{H-1} \tau_{m-j}, \quad H=3, \label{eq:tranche}\\
h_{m} &= -\sum_{g} b_{m,g}\,\hat\beta_{m,g}, \qquad x_m = (b_{m,1},\dots,b_{m,13},\,h_m), \label{eq:hedge}\\
k_m &= \min\!\left(\frac{0.10}{\sqrt{12\,x_m^{\top}\hat\Sigma_m x_m}},\ \frac{2}{\lVert x_m\rVert_1}\right), \qquad w_m = k_m\,x_m, \label{eq:scale}\\
r^{\mathrm{net}}_{m} &= \sum_{g} w_{m,g}\, r^{e}_{m,g} + w_{m,M}\,\mathrm{MKT}_m
  - c\sum_{g}\lvert w_{m,g}-\tilde w_{m,g}\rvert - c_h\,\lvert w_{m,M}-\tilde w_{m,M}\rvert. \label{eq:net}
\end{align}
where $s_{m,g}$ is the score of group $g$ known at the end of month $m-1$, $\hat\beta_{m,g}$ and the Ledoit--Wolf covariance
$\hat\Sigma_m$ (13 groups plus the market) use the 60 months ending in $m-2$ (at least 36), $r^{e}_{m,g}$ is the group's excess
return, $\tilde w$ are the previous weights after drift, $c=10$\,bp and $c_h=2$\,bp."""
(P.TEX / 'portfolio_equation.tex').write_text(equation + '\n')
print((P.TEX / 'portfolio_equation.tex').read_text()[:200])

% Score-to-weight mapping of the frozen study's backtest (230ga-follow-the-workers/outputs/follow_the_workers/data.py::backtest).
% Written by analysis/run.ipynb. Parameters: analysis/params.py PORTFO


---
##### Diagnostic helpers
Windows, IC statistics and a registry that feeds diagnostics_v2.md; everything below is POST-HOC unless labelled CONFIRMATORY.

In [7]:
SIGNALS = ['F1', 'F2', 'F3', 'F4', 'F5', 'W', 'A0', 'IndMom']
months = lambda w: pd.period_range(*P.WINDOWS[w], freq='M')
REGISTRY = []


def ic_stats(score, target, window_months, lags=P.HAC_LAGS):
    """Mean monthly rank IC in a window with its Newey-West s.e., t and number of months."""
    ic = S.rank_ic(score.reindex(window_months), target.reindex(window_months))
    h = S.hac(ic, lags=lags)
    return h['mean'], h['se'], h['t'], h['n']


def register(item, title, label, name, sources, df, note=''):
    REGISTRY.append(dict(item=item, title=title, label=label, name=name, sources=sources, df=df, note=note))


def ic_text(ic, t, digits=3):
    return [f'{a:+.{digits}f} ({b:+.2f})' if np.isfinite(a) else 'n/a' for a, b in zip(ic, t)]

---
##### D1 Single-signal IC
Rank IC of each labor signal, the preregistered composite and industry momentum against next-month relative returns.

In [8]:
rows = {}
for s in SIGNALS:
    row = {}
    for w in ['research', 'test', 'full']:
        m, se, t, n = ic_stats(F[s], R['relative'], months(w))
        row.update({f'{w}_ic': m, f'{w}_se': se, f'{w}_t': t, f'{w}_n': n})
    rows[s] = row
d1 = pd.DataFrame(rows).T
d1.index.name = 'signal'
d1_show = pd.DataFrame({'Research 2004--14': ic_text(d1.research_ic, d1.research_t),
                        'Test 2014--26': ic_text(d1.test_ic, d1.test_t),
                        'Full 2004--26': ic_text(d1.full_ic, d1.full_t)},
                       index=pd.Index([G.SIGNAL_LABELS[s] for s in d1.index], name='Signal'))
G.write_table(d1, 'd1_signal_ic', label=P.POSTHOC, display=d1_show, align='lrrr')
G.ic_signals(d1)
register('D1', 'Single-signal rank IC (t in brackets), next-month relative return', P.POSTHOC, 'd1_signal_ic',
         'analysis/data/fred (revised, fixed lags), analysis/data/french', d1_show,
         'IC = Spearman correlation across 13 groups each month; t from Newey-West with 6 lags. '
         'Industry momentum uses months m-12..m-2 (skips m-1), matching the frozen information rule. '
         'The 3 Oct baseline row "IndMom 12-1" used m-12..m-1 and is superseded; its "IndMom 12-2" row equals this definition. '
         f'Sensitivity: W standardisation history now starts at the frozen FEATURE_START (decision Apr 2002). The 3 Oct baseline '
         f'started it in Jan 2001 and found research/test IC +0.039/+0.041; here {d1.loc["W", "research_ic"]:+.3f}/{d1.loc["W", "test_ic"]:+.3f}. '
         'The full-sample IC is unchanged, but the research/test split is not stable to this choice.')
d1.round(3)

,research_ic,research_se,research_t,research_n,test_ic,test_se,test_t,test_n,full_ic,full_se,full_t,full_n
signal,,,,,,,,,,,,
F1,0.001,0.024,0.024,125.0,-0.011,0.026,-0.436,142.0,-0.005,0.018,-0.298,268.0
F2,0.012,0.025,0.481,125.0,0.013,0.028,0.478,142.0,0.013,0.019,0.710,268.0
F3,0.014,0.028,0.495,125.0,0.002,0.021,0.115,142.0,0.009,0.017,0.502,268.0
F4,0.033,0.030,1.070,125.0,0.026,0.028,0.932,142.0,0.028,0.020,1.376,268.0
F5,-0.012,0.021,-0.555,125.0,-0.007,0.020,-0.340,142.0,-0.010,0.015,-0.674,268.0
W,0.029,0.019,1.537,125.0,0.049,0.024,2.036,142.0,0.039,0.016,2.502,268.0
A0,-0.023,0.026,-0.879,125.0,-0.006,0.026,-0.212,142.0,-0.013,0.018,-0.689,268.0
IndMom,0.036,0.029,1.254,125.0,0.003,0.028,0.108,142.0,0.020,0.020,0.990,268.0


---
##### D2 IC by horizon
Rank IC of each signal against 1-, 3-, 6- and 12-month compounded relative returns, full sample.

In [9]:
d2_ic, d2_t = {}, {}
for h in P.HORIZONS:
    target = D.horizon_target(R['total'], h)
    for s in SIGNALS:
        m, se, t, n = ic_stats(F[s], target, months('full'), lags=h + 5)
        d2_ic.setdefault(s, {})[h] = m
        d2_t.setdefault(s, {})[h] = t
d2_ic, d2_t = pd.DataFrame(d2_ic).T, pd.DataFrame(d2_t).T
d2 = pd.concat([d2_ic.add_prefix('ic_h'), d2_t.add_prefix('t_h')], axis=1)
d2.index.name = 'signal'
d2_show = pd.DataFrame({f'{h}m': ic_text(d2_ic[h], d2_t[h]) for h in P.HORIZONS},
                       index=pd.Index([G.SIGNAL_LABELS[s] for s in d2_ic.index], name='Signal'))
G.write_table(d2, 'd2_horizon_ic', label=P.POSTHOC, display=d2_show, align='lrrrr')
G.horizon(d2_ic, d2_t)
register('D2', 'Rank IC by horizon h (t in brackets), May 2004-Aug 2026', P.POSTHOC, 'd2_horizon_ic',
         'analysis/data/fred, analysis/data/french', d2_show,
         'Target = h-month compounded total return demeaned across groups; Newey-West lags h+5 (frozen convention).')
d2.round(3)

,ic_h1,ic_h3,ic_h6,ic_h12,t_h1,t_h3,t_h6,t_h12
signal,,,,,,,,
F1,-0.005,0.007,-0.011,-0.039,-0.307,0.226,-0.284,-0.918
F2,0.013,0.002,0.002,-0.062,0.703,0.069,0.084,-2.081
F3,0.009,-0.013,-0.022,-0.065,0.500,-0.553,-0.775,-1.727
F4,0.028,0.005,0.000,-0.036,1.370,0.169,0.009,-1.277
F5,-0.010,-0.027,-0.049,-0.057,-0.672,-1.244,-1.689,-1.861
W,0.039,0.037,0.037,0.026,2.506,1.769,1.277,0.633
A0,-0.013,-0.002,-0.021,-0.053,-0.689,-0.085,-0.609,-1.580
IndMom,0.020,0.025,0.048,0.029,0.990,0.801,1.237,0.635


---
##### D3 Tightness
The tightness path, why expanding terciles fail, and signal IC in tightening vs loosening months.

In [10]:
G.timeline(T)
# Regime balance: the frozen expanding-tercile rule vs a 12-month change rule.
low, high = T.expanding(min_periods=P.STD_MIN_MONTHS).quantile(1 / 3), T.expanding(min_periods=P.STD_MIN_MONTHS).quantile(2 / 3)
tercile = pd.Series('middle', index=T.index).where(T.notna())
tercile[T <= low] = 'low'
tercile[T >= high] = 'high'
tercile[low.isna()] = np.nan
dT = T - T.shift(12)
balance = {}
for w in ['research', 'test']:
    idx = months(w)
    counts = tercile.reindex(idx).value_counts()
    balance[w] = {'months': len(idx), 'expanding low': int(counts.get('low', 0)), 'expanding middle': int(counts.get('middle', 0)),
                  'expanding high': int(counts.get('high', 0)), 'T rising (12m)': int((dT.reindex(idx) > 0).sum()),
                  'T falling (12m)': int((dT.reindex(idx) <= 0).sum()), 'V/U > 1': int((T.reindex(idx) > 0).sum()),
                  'T min': float(T.reindex(idx).min()), 'T max': float(T.reindex(idx).max())}
d3_balance = pd.DataFrame(balance).T
d3_balance.index.name = 'period'
d3_balance_show = pd.DataFrame({w: [f'{int(v)}' if k not in ('T min', 'T max') else f'{v:+.2f}' for k, v in d3_balance.loc[w].items()]
                                for w in d3_balance.index}, index=pd.Index(d3_balance.columns, name='Months in regime'))
d3_balance_show.columns = ['Research 2004--14', 'Test 2014--26']
G.write_table(d3_balance, 'd3_regime_balance', label=P.POSTHOC, display=d3_balance_show, align='lrr')
# IC by tightening / loosening, full sample (and by period in the CSV).
rows, by_period = {}, []
for s in SIGNALS:
    ic = S.rank_ic(F[s].reindex(months('full')), R['relative'].reindex(months('full')))
    up, down = dT.reindex(ic.index) > 0, dT.reindex(ic.index) <= 0
    r, f = S.hac(ic.where(up)), S.hac(ic.where(down))
    rows[s] = {'rising_ic': r['mean'], 'rising_se': r['se'], 'rising_t': r['t'], 'n_rising': r['n'],
               'falling_ic': f['mean'], 'falling_se': f['se'], 'falling_t': f['t'], 'n_falling': f['n'],
               'difference': r['mean'] - f['mean']}
    for w in ['research', 'test']:
        sub = ic.reindex(months(w))
        for state, mask in [('rising', up), ('falling', down)]:
            hh = S.hac(sub.where(mask.reindex(sub.index, fill_value=False)))
            by_period.append({'signal': s, 'period': w, 'state': state, 'ic': hh['mean'], 't': hh['t'], 'n': hh['n']})
d3 = pd.DataFrame(rows).T
d3.index.name = 'signal'
pd.DataFrame(by_period).to_csv(P.TABLES / 'd3_tightness_ic_by_period.csv', index=False)
d3_show = pd.DataFrame({f'T rising ({int(d3.n_rising.iloc[0])} m)': ic_text(d3.rising_ic, d3.rising_t),
                        f'T falling ({int(d3.n_falling.iloc[0])} m)': ic_text(d3.falling_ic, d3.falling_t),
                        'Difference': [f'{v:+.3f}' for v in d3.difference]},
                       index=pd.Index([G.SIGNAL_LABELS[s] for s in d3.index], name='Signal'))
G.write_table(d3, 'd3_tightness_ic', label=P.POSTHOC, display=d3_show, align='lrrr')
G.tightness(d3)
register('D3a', 'Tightness regime balance (months per regime)', P.POSTHOC, 'd3_regime_balance',
         'analysis/data/fred (JTSJOL, UNEMPLOY)', d3_balance_show,
         'Expanding terciles replicate the frozen tightness_regimes() rule (expanding quantiles, 24-month minimum).')
register('D3b', 'Rank IC in months when T rose vs fell over the previous 12 months, May 2004-Aug 2026', P.POSTHOC,
         'd3_tightness_ic', 'analysis/data/fred, analysis/data/french; per-period split in d3_tightness_ic_by_period.csv', d3_show)
d3_balance

,months,expanding low,expanding middle,expanding high,T rising (12m),T falling (12m),V/U > 1,T min,T max
period,,,,,,,,,
research,125.0,49.0,26.0,50.0,93.0,32.0,0.0,-1.878192,-0.304915
test,142.0,3.0,6.0,133.0,88.0,54.0,75.0,-1.611780,0.712436


---
##### D4 Cumulative test returns
CONFIRMATORY ledgers of the preregistered strategies, plus industry momentum through the same backtest (post-hoc).

In [11]:
conf_ledgers = {s: D.saved_csv(f'ASOF_{s}_ledger.csv') for s in ['A0', 'A1', 'A3', 'A4']}
mom_ledger, mom_weights = D.backtest(F['IndMom'].reindex(months('full')), R)   # continuous 2004-05..2026-08
test = months('test')
assert all(l.index.equals(test) for l in conf_ledgers.values())
names = {'A0': 'A0 fixed rule', 'A1': 'A1 ridge', 'A3': 'A3 primary', 'A4': 'A4 island agents'}
rows = {}
for s, l in conf_ledgers.items():
    rows[names[s]] = {'label': P.CONFIRMATORY, **S.perf(l)}
rows['Industry momentum 12-1'] = {'label': P.POSTHOC, **S.perf(mom_ledger.loc[test])}
d4 = pd.DataFrame(rows).T[['label', 'n', 'sharpe', 'net_ann', 'vol', 'max_dd', 'hit', 'turnover', 'cost_ann']]
d4.index.name = 'strategy'
series = pd.DataFrame({names[s]: l.net for s, l in conf_ledgers.items()} | {'Industry momentum 12-1': mom_ledger.net.loc[test]})
series.to_csv(P.TABLES / 'd4_test_net_returns.csv')
d4_show = d4.copy()
for c, f in {'sharpe': '{:+.2f}', 'net_ann': '{:+.1%}', 'vol': '{:.1%}', 'max_dd': '{:.1%}', 'hit': '{:.0%}', 'turnover': '{:.2f}', 'cost_ann': '{:.2%}'}.items():
    d4_show[c] = [f.format(v) for v in d4[c]]
d4_show.columns = ['Label', 'Months', 'Sharpe', 'Net p.a.', 'Vol.', 'Max DD', 'Hit rate', 'Turnover', 'Costs p.a.']
G.write_table(d4, 'd4_test_performance', label=f'{P.CONFIRMATORY} (A0-A4) / {P.POSTHOC} (momentum)',
              display=d4_show.drop(columns=['Label', 'Months', 'Hit rate']), align='lrrrrrr')
G.cumulative({names[s]: l.net for s, l in conf_ledgers.items()},
             {'Industry momentum 12-1 (post-hoc)': (mom_ledger.net.loc[test], ':')})
register('D4', 'Test-period performance, Nov 2014-Aug 2026', 'CONFIRMATORY (A0-A4) / POST-HOC (momentum)', 'd4_test_performance',
         '230ga-follow-the-workers/.../sealed/initial/ASOF_*_ledger.csv; momentum: frozen backtest on analysis/data/french', d4_show,
         'Annualised mean = 12 x monthly mean net excess return; Sharpe on net excess returns; drawdown of funded wealth (RF + net). '
         'Momentum backtest runs continuously from May 2004; the test window is a slice.')
d4

,label,n,sharpe,net_ann,vol,max_dd,hit,turnover,cost_ann
strategy,,,,,,,,,
A0 fixed rule,CONFIRMATORY,142,-0.359456,-0.034706,0.096553,-0.402753,0.464789,0.831768,0.01012
A1 ridge,CONFIRMATORY,142,-0.507481,-0.03988,0.078585,-0.377362,0.450704,0.910112,0.011066
A3 primary,CONFIRMATORY,142,-0.584495,-0.045877,0.078489,-0.410629,0.443662,0.913789,0.01111
A4 island agents,CONFIRMATORY,142,-0.786325,-0.060973,0.077542,-0.532903,0.464789,1.105963,0.013452
Industry momentum 12-1,POST-HOC,142,0.03484,0.003287,0.094335,-0.208265,0.471831,0.531534,0.00651


---
##### D5 Industry P&L
Where the primary A3 made and lost money in the test period.

In [12]:
w3, l3 = D.saved_csv('ASOF_A3_weights.csv'), D.saved_csv('ASOF_A3_ledger.csv')
groups = list(P.GROUPS)
contrib = (w3[groups] * R['excess'].loc[w3.index, groups]).sum()
pnl = pd.concat([contrib.rename(index=P.NAMES),
                 pd.Series({'Market hedge': float((w3['market'] * R['factors'].loc[w3.index, 'Mkt-RF']).sum()),
                            'Trading costs': -float(l3.cost.sum() + l3.borrow_financing.sum())})])
assert abs(pnl.sum() - l3.net.sum()) < 1e-10, 'P&L decomposition does not add up to net'
d5 = pd.DataFrame({'contribution': pnl,
                   'months_long_share': pd.Series({P.NAMES[g]: (w3[g] > 1e-9).mean() for g in groups}),
                   'months_short_share': pd.Series({P.NAMES[g]: (w3[g] < -1e-9).mean() for g in groups})}).reindex(pnl.index)
d5.index.name = 'component'
d5_show = pd.DataFrame({'Contribution (pp)': [f'{100 * v:+.1f}' for v in d5.contribution],
                        'Months long': ['' if np.isnan(v) else f'{v:.0%}' for v in d5.months_long_share],
                        'Months short': ['' if np.isnan(v) else f'{v:.0%}' for v in d5.months_short_share]}, index=d5.index)
G.write_table(d5, 'd5_industry_pnl', label=P.POSTHOC, display=d5_show, align='lrrr')
G.industry_pnl(pnl)
register('D5', 'A3 test-period P&L by industry (sum of monthly contributions)', P.POSTHOC, 'd5_industry_pnl',
         'sealed/initial/ASOF_A3_weights.csv, ASOF_A3_ledger.csv; analysis/data/french', d5_show,
         f'Components add up to the sum of A3 monthly net returns ({100 * l3.net.sum():+.1f} pp).')
d5.round(4)

,contribution,months_long_share,months_short_share
component,,,
Mining & logging,0.0721,0.3310,0.3592
Construction,-0.1918,0.3592,0.3873
Durable mfg,0.0176,0.3099,0.2394
Nondurable mfg,0.0542,0.3239,0.3451
Wholesale,0.1491,0.3099,0.4225
Retail,0.0023,0.3873,0.3451
Transport & utilities,0.0143,0.4718,0.2958
Information,-0.0583,0.3028,0.3732
Finance & insurance,-0.1630,0.3380,0.4507


---
##### D6 Factor loadings
CONFIRMATORY factor regressions of the preregistered books, read from the frozen results.json.

In [13]:
FACTORS = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom', 'group_momentum']
load_names = {'A0': 'A0 fixed rule', 'A1': 'A1 ridge', 'A3': 'A3 primary'}
rows = []
for s, label in load_names.items():
    a = res['attribution'][s]
    for f in ['intercept'] + FACTORS:
        rows.append({'strategy': label, 'factor': f, 'coef': a['coefficients'][f], 'se': a['standard_errors'][f],
                     't': a['t_statistics'][f], 'n': a['n'], 'label': P.CONFIRMATORY})
d6 = pd.DataFrame(rows)
d6_show = pd.DataFrame({lab: [(f'{100 * c:+.2f}% ({t:+.2f})' if f == 'intercept' else f'{c:+.2f} ({t:+.2f})')
                              for f, c, t in d6.loc[d6.strategy == lab, ['factor', 'coef', 't']].itertuples(index=False)]
                        for lab in load_names.values()},
                       index=pd.Index(['Alpha (per month)'] + ['Market', 'SMB', 'HML', 'RMW', 'CMA', 'Momentum', 'Industry momentum'], name='Factor'))
G.write_table(d6.set_index(['strategy', 'factor']), 'd6_factor_loadings', label=P.CONFIRMATORY, display=d6_show, align='lrrr')
G.loadings(d6[d6.factor != 'intercept'])
register('D6', 'Factor loadings of net long-short returns, test period (t in brackets)', P.CONFIRMATORY, 'd6_factor_loadings',
         '230ga-follow-the-workers/.../sealed/initial/results.json -> attribution', d6_show,
         'Newey-West, 6 lags. Gate 1 re-estimated these from the saved ledgers and matched to < 1e-8.')
d6_show

,A0 fixed rule,A1 ridge,A3 primary
Factor,,,
Alpha (per month),-0.25% (-0.92),-0.35% (-1.72),-0.38% (-1.86)
Market,-0.01 (-0.31),-0.01 (-0.20),-0.03 (-0.43)
SMB,+0.02 (+0.21),+0.12 (+1.01),+0.10 (+0.89)
HML,+0.09 (+0.60),-0.15 (-1.97),-0.15 (-2.01)
RMW,+0.00 (+0.05),+0.21 (+2.33),+0.19 (+2.16)
CMA,-0.04 (-0.21),+0.05 (+0.47),+0.03 (+0.26)
Momentum,-0.06 (-0.72),+0.01 (+0.15),-0.01 (-0.17)
Industry momentum,+0.22 (+1.93),-0.02 (-0.21),-0.01 (-0.12)


---
##### D7 Agent features
Research IC vs test IC of every valid repetition-1 candidate in both studies; selected and regime-gated ones marked.

In [14]:
decode = {v: k for k, v in json.loads((P.SONNET / 'outputs/blind_map.json').read_text())['variables'].items()}
pretty = {'JOR': 'Openings', 'HIR': 'Hires', 'QUR': 'Quits', 'LDR': 'Layoffs', 'H': 'Hours', 'E': 'Earnings'}
rows = []
for study, root in [('Sonnet', P.SONNET), ('Opus', P.OPUS)]:
    sel = json.loads((root / 'outputs/selected_features.json').read_text())
    out = json.loads((root / 'outputs/sealed/initial/results.json').read_text())
    specs = {x['candidate_id']: x for x in sel['seed_one_specs']}
    chosen = {c for v in sel['selected'].values() for c in v}
    for a in out['candidate_accounting']:
        cid = a['candidate_id']
        if a.get('valid') is False:
            rows.append({'study': study, 'candidate': cid, 'valid': False})
            continue
        expr = specs[cid]['spec']['expression']
        decoded = expr
        for vid, name in decode.items():
            decoded = decoded.replace(vid, pretty[name])
        rows.append({'study': study, 'candidate': cid, 'valid': True, 'arm': specs[cid]['arm'], 'expression': expr,
                     'decoded': decoded, 'research_ic': a['research']['mean'], 'research_t': a['research']['t'],
                     'research_n': a['research']['n'], 'test_ic': a['sealed']['mean'], 'test_t': a['sealed']['t'],
                     'test_n': a['sealed']['n'], 'passed_research': a['passed_research_filter'],
                     'selected': cid in chosen, 'gated': 'regime(' in expr})
d7 = pd.DataFrame(rows).astype({'research_ic': float, 'test_ic': float})
valid = d7[d7.valid == True].copy()
G.agents_scatter(valid)
summary = {}
for study in ['Sonnet', 'Opus']:
    v = valid[valid.study == study]
    both = v.dropna(subset=['research_ic', 'test_ic'])
    summary[study] = {'valid candidates': len(v), 'regime-gated': int(v.gated.sum()),
                      'test IC undefined (<=3 active months)': int(v.test_ic.isna().sum()),
                      'corr(research IC, test IC)': float(both.research_ic.corr(both.test_ic)),
                      'passed research filter': int(v.passed_research.sum()), 'selected': int(v.selected.sum()),
                      'selected and regime-gated': int((v.selected & v.gated).sum()),
                      'selected with positive test IC': int((v.selected & (v.test_ic > 0)).sum())}
d7_summary = pd.DataFrame(summary)
d7_summary.index.name = 'item'
d7.to_csv(P.TABLES / 'd7_agent_candidates.csv', index=False)
chosen_rows = valid[valid.selected].copy()
d7_show = pd.DataFrame({'Study': chosen_rows.study, 'Arm': chosen_rows.arm, 'Feature (decoded)': chosen_rows.decoded.str.replace(',', ', '),
                        'Research IC (n)': [f'{a:+.3f} ({int(n)})' for a, n in zip(chosen_rows.research_ic, chosen_rows.research_n)],
                        'Test IC (n)': [('n/a' if not np.isfinite(a) else f'{a:+.3f}') + f' ({int(n)})' for a, n in zip(chosen_rows.test_ic, chosen_rows.test_n)]})
d7_summary_show = d7_summary.apply(lambda row: [f'{v:+.2f}' if row.name.startswith('corr') else f'{int(v)}' for v in row], axis=1, result_type='broadcast').astype(str)
d7_summary_show.index = [i.replace('<=', '≤') for i in d7_summary_show.index]
d7_summary_show.index.name = 'Repetition-1 candidates'
G.write_table(d7_summary, 'd7_agent_summary', label=P.POSTHOC, display=d7_summary_show, align='lrr')
G.write_table(chosen_rows.set_index('candidate')[['study', 'arm', 'decoded', 'research_ic', 'research_n', 'test_ic', 'test_n', 'gated']],
              'd7_selected_features', label=P.POSTHOC, display=d7_show.set_index('Study'), align='llp{2.55in}rr')
register('D7a', 'Agent candidates, research vs test (repetition 1, both studies)', P.POSTHOC, 'd7_agent_summary',
         'sealed/initial/results.json -> candidate_accounting; selected_features.json (both studies)', d7_summary_show)
register('D7b', 'Selected agent features, decoded after evaluation', P.POSTHOC, 'd7_selected_features',
         'selected_features.json, results.json, blind_map.json (both studies)', d7_show.set_index('Study'),
         'Full candidate list: analysis/output/tables/d7_agent_candidates.csv. Test IC is undefined when a feature is active in 3 or fewer months.')
d7_summary

,Sonnet,Opus
item,,
valid candidates,33.000000,35.000000
regime-gated,9.000000,6.000000
test IC undefined (<=3 active months),4.000000,2.000000
"corr(research IC, test IC)",0.224763,0.372355
passed research filter,4.000000,7.000000
selected,4.000000,4.000000
selected and regime-gated,2.000000,2.000000
selected with positive test IC,0.000000,2.000000


---
##### D8 A3 windows
CONFIRMATORY window statistics of the primary strategy from the frozen results.json.

In [15]:
win = res['windows']['A3']
labels8 = {'whole': 'Full test, Nov 2014-Aug 2026', 'first_half': 'First half', 'second_half': 'Second half',
           'ex_pandemic': 'Excluding Mar-Dec 2020', 'last_18': 'Last 18 months (Mar 2025-Aug 2026)'}
rows = {labels8[k]: {'months': win[k]['n'], 'sharpe': win[k]['sharpe'], 'net_ann': win[k]['mean_net_annual'],
                     'ic': win[k]['IC']['mean'], 'ic_t': win[k]['IC']['t'], 'scope': 'test'} for k in labels8}
for k, lab in [('full', 'Course window "full" (from Jun 2007)'), ('post_2010', 'Course window "post-2010"')]:
    c = res['course_windows']['A3'][k]
    rows[lab] = {'months': c['n'], 'sharpe': c['sharpe'], 'net_ann': c['mean_net_annual'], 'ic': c['IC']['mean'],
                 'ic_t': c['IC']['t'], 'scope': 'research (pseudo-real-time) + test, descriptive'}
d8 = pd.DataFrame(rows).T
d8.index.name = 'window'
d8_show = pd.DataFrame({'Months': d8.months.astype(int), 'Sharpe': [f'{v:+.2f}' for v in d8.sharpe],
                        'Net return p.a.': [f'{v:+.1%}' for v in d8.net_ann], 'Mean IC (t)': ic_text(d8.ic.astype(float), d8.ic_t.astype(float))},
                       index=d8.index)
G.write_table(d8, 'd8_a3_windows', label=P.CONFIRMATORY, display=d8_show, align='lrrrr')
register('D8', 'Primary A3 by window', P.CONFIRMATORY, 'd8_a3_windows',
         '230ga-follow-the-workers/.../sealed/initial/results.json -> windows.A3, course_windows.A3', d8_show,
         'Course windows mix the pseudo-real-time research period with the test period; the frozen study labels them descriptive.')
d8_show

,Months,Sharpe,Net return p.a.,Mean IC (t)
window,,,,
"Full test, Nov 2014-Aug 2026",142,-0.58,-4.6%,-0.021 (-1.08)
First half,71,-0.72,-4.9%,-0.028 (-0.90)
Second half,71,-0.49,-4.3%,-0.015 (-0.59)
Excluding Mar-Dec 2020,132,-0.51,-4.1%,-0.012 (-0.56)
Last 18 months (Mar 2025-Aug 2026),18,+0.85,+6.7%,+0.033 (+0.44)
"Course window ""full"" (from Jun 2007)",231,-0.44,-3.2%,-0.009 (-0.49)
"Course window ""post-2010""",200,-0.62,-4.4%,-0.019 (-1.02)


---
##### D9 Risk overlay
How often the gross cap bound and how ex-ante volatility compares with realised volatility, from the saved test ledgers.

In [16]:
rows = []
for study, root, strategies in [('Sonnet', P.SONNET, ['A0', 'A1', 'A1-T', 'A3', 'A4']), ('Opus', P.OPUS, ['A3', 'A4'])]:
    for s in strategies:
        l = D.saved_csv(f'ASOF_{s}_ledger.csv', root)
        rows.append({'study': study, 'strategy': s, 'months': len(l),
                     'cap_binding_share': float((l.gross_leverage >= P.PORTFOLIO['gross_cap'] - 1e-3).mean()),
                     'mean_ex_ante_vol': float(l.ex_ante_vol.mean()), 'realised_vol': float(l.net.std() * np.sqrt(12)),
                     'mean_gross_leverage': float(l.gross_leverage.mean()), 'mean_abs_hedge': float(l.market_hedge.abs().mean())})
d9 = pd.DataFrame(rows).set_index(['study', 'strategy'])
d9_show = pd.DataFrame({'Cap binds': [f'{v:.0%}' for v in d9.cap_binding_share], 'Ex-ante vol': [f'{v:.1%}' for v in d9.mean_ex_ante_vol],
                        'Realised vol': [f'{v:.1%}' for v in d9.realised_vol], 'Gross leverage': [f'{v:.2f}' for v in d9.mean_gross_leverage],
                        '|Hedge|': [f'{v:.2f}' for v in d9.mean_abs_hedge]},
                       index=pd.Index([f'{a} {b}' for a, b in d9.index], name='Study, strategy'))
G.write_table(d9, 'd9_risk_overlay', label=P.POSTHOC, display=d9_show, align='lrrrrr')
register('D9', 'Risk overlay in the test period (target volatility 10%, gross cap 2)', P.POSTHOC, 'd9_risk_overlay',
         'sealed/initial/ASOF_*_ledger.csv (both studies)', d9_show,
         'Cap binds = share of months with gross leverage >= 1.999. Opus A0/A1/A1-T are identical to Sonnet and omitted.')
d9_show

,Cap binds,Ex-ante vol,Realised vol,Gross leverage,|Hedge|
"Study, strategy",,,,,
Sonnet A0,80%,8.0%,9.7%,1.96,0.15
Sonnet A1,88%,7.5%,7.9%,1.98,0.13
Sonnet A1-T,82%,7.5%,8.2%,1.97,0.12
Sonnet A3,87%,7.5%,7.8%,1.98,0.13
Sonnet A4,91%,7.3%,7.8%,1.99,0.13
Opus A3,92%,7.3%,7.5%,1.99,0.15
Opus A4,93%,7.3%,8.4%,1.99,0.11


---
##### D10 Agent experiment design
Diagram and parameter table of the blinded agent experiment, read from the frozen study.

In [17]:
G.agents_diagram()
design_rows = []
for study, root in [('Sonnet', P.SONNET), ('Opus', P.OPUS)]:
    cand = pd.read_csv(root / 'outputs/candidates.csv')
    mig = json.loads((root / 'outputs/agents/migration_summary.json').read_text())
    hr = json.loads((root / 'outputs/human_review_submission.json').read_text())
    items = hr['items']
    design_rows += [
        (study, 'Valid proposals', f"{int(cand.valid.sum())} / {len(cand)}", 'outputs/candidates.csv'),
        (study, 'Migration reports admitted', f"{mig['reports_admitted']} / {mig['reports_total']}", 'outputs/agents/migration_summary.json'),
        (study, 'Defined LLM-judge uptake labels', f"{mig['defined_judgments']} / {mig['n']}", 'outputs/agents/migration_summary.json'),
        (study, 'Human-audited judge pairs', f"{len(items)}", 'outputs/human_review_submission.json')]
    if 'agreement_count' in hr:
        design_rows.append((study, 'Human-judge agreement', f"{hr['agreement_count']} / {hr['reviewed']} (human yes {hr['human_yes']}, judge yes {hr['machine_yes_in_sample']})",
                            'outputs/human_review_submission.json'))
A = FP.AGENT
design_rows = [('Both', 'Arms', 'islands (communicating), independent (control)', 'params.py AGENT'),
               ('Both', 'Agents per arm', ', '.join(A['emphases']), 'params.py AGENT'),
               ('Both', 'Proposals per agent x repetitions', f"{A['submissions']} x {len(A['seeds'])}", 'params.py AGENT'),
               ('Both', 'Migration reports', f"after proposals {' and '.join(map(str, A['migration_after']))}, at most {A['migration_word_limit']} words", 'params.py AGENT'),
               ('Both', 'Selection (repetition 1 only)', f"t ≥ {FP.MODEL['candidate_min_t']}, same sign in both halves, |corr| < {FP.MODEL['candidate_max_correlation']}, at most {FP.MODEL['candidate_max']} per arm", 'params.py MODEL')] + design_rows
d10 = pd.DataFrame(design_rows, columns=['study', 'item', 'value', 'source'])
G.write_table(d10, 'd10_agent_design', index=False, label=P.CONFIRMATORY, display=d10[['study', 'item', 'value']], align='llp{3.3in}')
register('D10', 'Agent experiment design and process counts', P.CONFIRMATORY, 'd10_agent_design',
         'frozen params.py, candidates.csv, migration_summary.json, human_review_submission.json (both studies)', d10.set_index('study'))
d10

,study,item,value,source
0,Both,Arms,"islands (communicating), independent (control)",params.py AGENT
1,Both,Agents per arm,"Temporal, Composition, Interactions",params.py AGENT
2,Both,Proposals per agent x repetitions,6 x 3,params.py AGENT
3,Both,Migration reports,"after proposals 2 and 4, at most 120 words",params.py AGENT
4,Both,Selection (repetition 1 only),"t ≥ 1.5, same sign in both halves, |corr| < 0....",params.py MODEL
5,Sonnet,Valid proposals,98 / 108,outputs/candidates.csv
6,Sonnet,Migration reports admitted,7 / 18,outputs/agents/migration_summary.json
7,Sonnet,Defined LLM-judge uptake labels,8 / 36,outputs/agents/migration_summary.json
8,Sonnet,Human-audited judge pairs,8,outputs/human_review_submission.json
9,Opus,Valid proposals,106 / 108,outputs/candidates.csv


---
##### Diagnostics report
Write analysis/output/diagnostics_v2.md with every number and the file it came from.

In [18]:
figures = sorted(p.name for p in P.FIGS.glob('fig_*.pdf'))
parts = [f'# Diagnostics v2 (Part 2)\n\nGenerated by analysis/run.ipynb on {utc_now()}. Every number below is copied from the CSV '
         'named in its header (analysis/output/tables/), which the notebook writes from the sources listed. Labels follow CLAUDE.md. '
         'Historical labor signals use revised FRED data with fixed lags (see data_vintage.md). Figures (vector PDF): '
         + ', '.join(f'`report/figures/{f}`' for f in figures) + '.\n']
for r in REGISTRY:
    parts += [f"## {r['item']}. {r['title']}\n", f"Label: **{r['label']}**. File: `analysis/output/tables/{r['name']}.csv` "
              f"(LaTeX: `report/tables/{r['name']}.tex`). Source: {r['sources']}.\n", G.markdown(r['df']), '']
    if r['note']:
        parts += [r['note'], '']
(P.OUT / 'diagnostics_v2.md').write_text('\n'.join(parts))
print('figures:', figures)

figures: ['fig_agents.pdf', 'fig_agents_scatter.pdf', 'fig_cumulative.pdf', 'fig_horizon.pdf', 'fig_ic_signals.pdf', 'fig_industry_pnl.pdf', 'fig_loadings.pdf', 'fig_tightness.pdf', 'fig_timeline.pdf']
